# VFD Crystallisation — Benchmark Comparison

Fair comparison of four models on identical setups:
- M1: Lindblad (standard open quantum)
- M2: GRW (stochastic collapse)
- M3: OR (Penrose-inspired)
- M4: VFD Crystallisation

In [ ]:
import sys
sys.path.insert(0, '../src')
sys.path.insert(0, '../benchmarks')

import numpy as np
import matplotlib.pyplot as plt
from crystallisation_benchmark import (
    make_qubit_config, make_biased_config, make_multilevel_config,
    make_noise_sweep_config, run_benchmark, summarise_benchmark,
    BenchmarkConfig,
)
from vfd.crystallisation.benchmark_metrics import (
    coherence_profile, purity_profile, basin_preference_index,
    bootstrap_ci,
)

plt.rcParams.update({'font.size': 11, 'figure.figsize': (12, 5)})
COLORS = {'Lindblad': '#B2182B', 'GRW': '#D6604D', 'OR': '#E08214', 'VFD': '#2166AC'}

## S1 — Single Qubit Benchmark

In [ ]:
config = make_qubit_config()
config = BenchmarkConfig(**{**config.__dict__, 'n_trials': 30, 'T': 3.0})
result = run_benchmark(config)
print(summarise_benchmark(result))

In [ ]:
# PLOT-1: Entropy comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
models = list(result.results.keys())
entropies = [result.results[m].metrics['outcome_entropy'] for m in models]
repeats = [result.results[m].metrics['repeatability'] for m in models]
colors = [COLORS[m] for m in models]

axes[0].bar(models, entropies, color=colors, alpha=0.85)
axes[0].set_ylabel('Outcome entropy H')
axes[0].set_title('PLOT-1: Outcome Entropy')
axes[0].grid(True, alpha=0.3)

axes[1].bar(models, repeats, color=colors, alpha=0.85)
axes[1].set_ylabel('Repeatability R')
axes[1].set_title('PLOT-2: Repeatability')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# PLOT-3: Transition time distributions
fig, ax = plt.subplots(figsize=(10, 4))
for m in models:
    times = result.results[m].transition_times
    ax.hist(times, bins=15, alpha=0.5, label=m, color=COLORS[m], density=True)
ax.set_xlabel('Transition time')
ax.set_ylabel('Density')
ax.set_title('PLOT-3: Transition Time Distributions')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# PLOT-4: Trajectory comparison (coherence decay)
fig, ax = plt.subplots(figsize=(10, 4))
for m in models:
    traj = result.results[m].trajectories[0]  # first trial
    cp = coherence_profile(traj)
    ax.plot(cp, label=m, color=COLORS[m], linewidth=1.2)
ax.set_xlabel('Time step')
ax.set_ylabel('Off-diagonal norm')
ax.set_title('PLOT-4: Coherence Decay (single trajectory)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## S3 — Multi-Level System

In [ ]:
config_ml = make_multilevel_config(4)
config_ml = BenchmarkConfig(**{**config_ml.__dict__, 'n_trials': 20, 'T': 3.0})
result_ml = run_benchmark(config_ml)

# PLOT-5: Basin selection
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
for ax, m in zip(axes, models):
    bpi = result_ml.results[m].metrics['basin_preference']
    ax.bar(range(4), bpi, color=COLORS[m], alpha=0.85)
    ax.set_xlabel('Basin')
    ax.set_title(m)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Preference')
plt.suptitle('PLOT-5: Basin Selection (4-level system)', fontsize=13)
plt.tight_layout()
plt.show()

## S5 — Noise Sweep

In [ ]:
gammas = [0.001, 0.01, 0.05, 0.1, 0.3, 0.5]
sweep_results = {}
for g in gammas:
    cfg = make_noise_sweep_config(g)
    cfg = BenchmarkConfig(**{**cfg.__dict__, 'n_trials': 15, 'T': 3.0})
    sweep_results[g] = run_benchmark(cfg)

# PLOT-6: VFD repeatability vs noise
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for m in models:
    R_vals = [sweep_results[g].results[m].metrics['repeatability'] for g in gammas]
    ax1.plot(gammas, R_vals, 'o-', label=m, color=COLORS[m], markersize=5)
ax1.set_xlabel(r'$\Gamma$ (environment coupling)')
ax1.set_ylabel('Repeatability')
ax1.set_title('Repeatability vs noise')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

for m in models:
    H_vals = [sweep_results[g].results[m].metrics['outcome_entropy'] for g in gammas]
    ax2.plot(gammas, H_vals, 'o-', label=m, color=COLORS[m], markersize=5)
ax2.set_xlabel(r'$\Gamma$ (environment coupling)')
ax2.set_ylabel('Outcome entropy')
ax2.set_title('Entropy vs noise')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Statistical Significance

In [ ]:
print('=== Pairwise comparisons (qubit) ===')
for comp_name, comp in result.comparisons.items():
    print(f'{comp_name}:')
    print(f'  KS stat={comp["ks_statistic"]:.4f}, p={comp["ks_pvalue"]:.4f}')
    print(f'  Entropy: VFD={comp["entropy_vfd"]:.4f} vs baseline={comp["entropy_baseline"]:.4f}')
    print(f'  Repeat:  VFD={comp["repeatability_vfd"]:.4f} vs baseline={comp["repeatability_baseline"]:.4f}')